# 15 — Diff-trigger q-transfer (interactive)

Transplant the goal-head **query** from SRC into DST at the pivot, and see what DST writes.
SRC and DST share two sub-tasks {A,B}; SRC has a unique **C**, DST a unique **D**:

```
order_matched:  SRC=[A,B,C] prefill{A,B} (frontier C)  |  DST=[A,B,D] prefill{A} (native B)
order_diff:     SRC=[A,C,B] prefill{A,B} (frontier C)  |  DST=[A,B,D] prefill{A} (native B)
```
Outcomes: **D** = abstract 'do the remaining given {A,B}' DIFF · **B** = native (no transfer) · **C** = literal copy (C isn't in DST's spec).

**Two knobs:** `PIVOT` (`indent` | `var_eq`) and `ALPHA` (OV-boost of the goal heads on the patched gen/readout). Set them in cell 3, run cell 4.

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
while not os.path.isdir('src') and os.getcwd() != '/':
    os.chdir('..')
sys.path.append('.')
import json, torch
MODEL = 'Qwen/Qwen2.5-Coder-32B-Instruct'
HEADS = 'head_sets/Qwen2.5-Coder-32B-Instruct/goal_topk/goal__lam20_top250.json'  # 250 goal heads
# HEADS = "head_sets/Qwen2.5-Coder-32B-Instruct/goal__lam20_thr0.9_band25-56.json"
os.environ['CUDA_VISIBLE_DEVICES'] = '3'   # set free GPU(s)
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

import src.diff.data as D, src.diff.functional as FUNC, src.diff.eval as E
ctx = FUNC.setup(MODEL); mt = ctx.mt
patch_heads = [(int(l), int(h)) for l, h in json.load(open(HEADS))]
print(f'{len(patch_heads)} goal heads loaded')

2026-09-18 01:52:35 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228
250 goal heads loaded


## Knobs

In [8]:
# ============================= KNOBS =============================
PIVOT = 'var_eq'      # 'indent' (bare trailing indent)  |  'var_eq' (the frontier's `{var} =` stub)
ALPHA = 2.0           # OV boost of the goal heads on the patched gen + readout (1.0 = plain patch)
RESTRICT_TASK = True # blind the goal heads to everything EXCEPT the task span (task list only)
SRC_LANG = 'python'   # 'python' | 'cpp'  — cross-language: SRC's language (query source)
DST_LANG = 'python'   # 'python' | 'cpp'  — DST's language (the run measured)
# ================================================================

# instance (order_matched `_mk` form): shared A,B ; SRC's unique C ; DST's unique D
# DATATYPE = 'list[str]'
DATATYPE = "int"
A = "absolute_value"
B = "is_even"
C = "is_perfect_square"
Dd = "count_set_bits"

VARIANT  = 'order_matched'   # 'order_matched' | 'order_diff'
# A, B = 'count_long', 'count_short'        # T1, T2  (shared)
# C    = 'distinct_count'                   # T3      (SRC's target)
# Dd   = 'count_ends_consonant'             # T4      (DST's remaining / DIFF target)
# A, B = "shortest_word_length", "count_starts_vowel"        # T1, T2  (shared)
# C    = "all_unique_lengths"                   # T3      (SRC's target)
# Dd   = "longest_word_length"             # T4      (DST's remaining / DIFF target)
# A  = "count_long"            # shared A
# B  = "count_short"           # native ("words shorter than 3 chars")   ← length op
# C  = "distinct_count"        # SRC's target ("distinct words")         ← uniqueness op
# Dd = "count_ends_consonant"  # DST's D ("words ending in a consonant")  ← last-char op
# A  = "distinct_count"      # shared A
# B  = "total_uppercase_letters"       # native ("length of the longest word")     ← length op
# Dd  = "total_length"   # SRC's target ("total uppercase letters")   ← char-count op
# C = "total_vowels"                 # DST's D ("whether any word is empty")      ← emptiness op


# VARIANT = 'order_diff'   # 'order_matched' | 'order_diff'
# A="count_ends_consonant"
# B="has_palindrome"
# C="longest_word_length" 
# Dd="total_vowels"


# available predicate ids for this datatype:
print([p.id for p in D.QTASKS[DATATYPE].predicates])

['is_even', 'is_positive', 'digit_count', 'is_multiple_of_3', 'is_two_digit', 'is_prime', 'is_perfect_square', 'is_power_of_two', 'digit_sum', 'is_palindrome', 'last_digit', 'is_odd', 'absolute_value', 'sign', 'is_negative', 'count_set_bits']


In [9]:
from rich import print as rich_print
from rich.markup import escape
from src.diff.functional import show_delta_tokens
from src.tokens import prepare_input
import numpy as np

def build_instance(datatype, A, B, C, Dd, pivot, variant='order_matched', src_lang='python', dst_lang='python'):
    bank = D.QTASKS[datatype]; by = {p.id: p for p in bank.predicates}
    a, b, c, d = by[A], by[B], by[C], by[Dd]
    src_order = (a, b, c) if variant == 'order_matched' else (a, c, b)   # order_diff: C listed 2nd
    return D.DiffTriggerInstance(
        datatype=datatype, src_order=src_order, dst_order=(a, b, d),
        src_progress=[A, B], dst_progress=[A], diff_target=Dd, dst_native=B, src_target=C,
        var_name=bank.var_names[0], presentation='listed', src_lang=src_lang, dst_lang=dst_lang,
        naming='letters', letter_stub=('var_eq' if pivot == 'var_eq' else 'indent'),
        meta={'variant': variant, 'dst_next': B, 'diff_target': Dd, 'src_target': C, 'datatype': datatype})

def run_case(inst, alpha, restrict_to_task=False):
    raw = FUNC.run_q_transfer_intervention(mt, inst, patch_heads, readout_heads=patch_heads,
                                           n_steer=16, max_new=96, alpha=alpha, n_pivot=1,
                                           restrict_to_task=restrict_to_task)
    causal = E.judge_causal(inst, raw['gen_patched'], model_name='claude-sonnet-4-6')
    lit = E.classify_concept(raw['gen_patched'], inst.src_target_desc, inst.dst_next_desc, inst.task_desc,
                             model_name='claude-sonnet-4-6',
                             kind='the code the model generated after the pivot; judge ONLY the first computation')
    return raw, causal, lit

def tail(p):
    return p[p.index('<|im_start|>user'):]

## Run

In [10]:
inst = build_instance(DATATYPE, A, B, C, Dd, PIVOT, VARIANT, SRC_LANG, DST_LANG)
raw, causal, lit = run_case(inst, ALPHA, RESTRICT_TASK)

print(f'### {VARIANT} | pivot={PIVOT} | alpha={ALPHA} | {SRC_LANG}→{DST_LANG} | restrict_to_task={RESTRICT_TASK} | {len(patch_heads)} heads ###')
print(f'shared A,B = {A}, {B}   |   SRC C = {C} ({inst.src_target_desc})')
print(f'DST native B = {B} ({inst.dst_next_desc})   |   DIFF target D = {Dd} ({inst.expected_next_desc})\n')

# ---- generations + goalscope readouts, one block per run (04-style) ----
rich_print(f"[red1]source_run · clean SRC (frontier C={C}):[/red1]\n{escape(raw['gen_src'])}")
rich_print(f"   [grey58]↳ goalscope · source_run:[/grey58] {escape(raw['ro_src'])}\n")
rich_print(f"[sea_green3]destination_run · clean DST (native B={B}):[/sea_green3]\n{escape(raw['gen_dst'])}")
rich_print(f"   [grey58]↳ goalscope · destination_run:[/grey58] {escape(raw['ro_dst'])}\n")
rich_print(f"[salmon1]patched_run · DST (expect DIFF D={Dd}):[/salmon1]\n{escape(raw['gen_patched'])}")
rich_print(f"   [grey58]↳ goalscope · patched_run:[/grey58] {escape(raw['ro_patched'])}\n")

# ---- ATTN_VIS · pivot→TASK span (goal heads), for the destination & patched runs ----
tp = raw['task_positions']
dst_ids = prepare_input(prompts=raw['dst_prompt'], tokenizer=mt)['input_ids'][0]
task_toks = [mt.tokenizer.decode([int(dst_ids[j])]) for j in tp]
show_delta_tokens(task_toks, np.asarray(raw['attn_clean']),   "ATTN_VIS · destination_run (clean)")
show_delta_tokens(task_toks, np.asarray(raw['attn_patched']), "ATTN_VIS · patched_run")
show_delta_tokens(task_toks, np.asarray(raw['attn_delta']),   "ATTN_DIFF · patched − clean")

# ---- verdict ----
trig = {'src':'D (DIFF ✓)','dst':'B (native ✗)','neither':'neither','both':'both'}.get(causal['label'], causal['label'])
print(f">>> causal: wrote {trig}   triggered(D)={causal['triggered']}   literal-C={'YES' if lit['label'] in ('src','both') else 'no'}")
print(f">>> why: {causal['why']}")

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=122 ' =' => I'm checking if the current number is a perfect square by comparing it to the square root of the 
number rounded to the nearest integer squared.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=112 ' =' => I'm checking if the number is even by using the modulus operator (`%`). The expression `(number %
2 == 0)` evaluates to `True` if the number is even, otherwise `False`.

GoalScope[additive_norm]:   0%|          | 0/1 [00:00<?, ?it/s]

pos=112 ' =' => I'm counting the number of 1s in the binary representation of the number (i.e., the number of set
bits). This can be done using the `bin()` function and `count('1')` method: `bin(x).count('1')`.

... (7 outputs left out)

ATTN_VIS · destination_run (clean)   (green +Δ / red −Δ, |Δ|max=1.137)

ATTN_VIS · patched_run   (green +Δ / red −Δ, |Δ|max=1.082)

ATTN_DIFF · patched − clean   (green +Δ / red −Δ, |Δ|max=0.929)

>>> causal: wrote D (DIFF ✓)   triggered(D)=True   literal-C=no
>>> why: The first actual computation performed is `bin(a).count('1')`, which counts set bits in the binary representation — that is sub-goal A.
